# Decision Trees from Scratch

Building a Decision Tree classifier **from scratch**.

We'll work through the pieces one at a time:
1. Measuring impurity (Entropy & Gini)
2. Finding the best split (Information Gain)
3. Building the tree recursively
4. Making predictions
5. Evaluating and visualising the result
6. We work with continuous variables only as of now

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

# For loading a dataset at the end
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)
print('Imports OK')

/Users/shivin/miniconda/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Imports OK


---
## Part 1 — Measuring Impurity

A decision tree splits data to make the resulting groups as **pure** as possible (i.e. one class dominates).

We need a number that measures how *mixed* a group is. Two common choices:

### 1a. Entropy

$$H(S) = -\sum_{k} p_k \log_2 p_k$$

where $p_k$ is the fraction of samples that belong to class $k$.

- **Entropy = 0** → perfectly pure (all one class)
- **Entropy = 1** (for 2 classes) → maximally mixed (50/50 split)

### 1b. Gini Impurity

$$G(S) = 1 - \sum_{k} p_k^2$$

Faster to compute (no logarithm), used by sklearn's CART by default.

In [2]:
def entropy(y):
    """Compute Shannon entropy of a label array y."""
    n = len(y)
    if n == 0:
        return 0.0
    counts = np.bincount(y)          # count occurrences of each class
    probs = counts[counts > 0] / n   # ignore zero-count classes (log(0) undefined)
    return -np.sum(probs * np.log2(probs))


def gini(y):
    """Compute Gini impurity of a label array y."""
    n = len(y)
    if n == 0:
        return 0.0
    counts = np.bincount(y)
    probs = counts / n
    return 1.0 - np.sum(probs ** 2)


# --- Quick sanity checks ---
pure   = np.array([0, 0, 0, 0])    # all same class
mixed  = np.array([0, 0, 1, 1])    # 50/50
skewed = np.array([0, 0, 0, 1])    # 75/25

print(f'Entropy  — pure: {entropy(pure):.3f}, mixed: {entropy(mixed):.3f}, skewed: {entropy(skewed):.3f}')
print(f'Gini     — pure: {gini(pure):.3f},   mixed: {gini(mixed):.3f},   skewed: {gini(skewed):.3f}')

Entropy  — pure: -0.000, mixed: 1.000, skewed: 0.811
Gini     — pure: 0.000,   mixed: 0.500,   skewed: 0.375


### Exercise 1
For an array with **3 classes equally represented** (e.g. `[0,0,1,1,2,2]`), compute entropy and Gini by hand, then verify with the functions above.

In [3]:
# Your code here
three_class = np.array([0, 0, 1, 1, 2, 2])
# Expected entropy ≈ log2(3) ≈ 1.585
# Expected gini   = 1 - 3*(1/3)^2 = 1 - 1/3 ≈ 0.667
print(entropy(three_class), gini(three_class))

1.584962500721156 0.6666666666666667


---
## Part 2 — Information Gain

Given a dataset $S$ with impurity $H(S)$, we split on feature $j$ at threshold $t$:
- **Left** child: all rows where $x_j \le t$
- **Right** child: all rows where $x_j > t$

The **Information Gain** measures how much the split *reduces* impurity:

$$\text{IG}(S, j, t) = H(S) - \frac{|S_L|}{|S|}H(S_L) - \frac{|S_R|}{|S|}H(S_R)$$

We want to find the $(j, t)$ pair that **maximises** IG.

In [4]:
def information_gain(y, y_left, y_right, criterion='entropy'):
    """Information gain from splitting y into y_left and y_right."""
    impurity_fn = entropy if criterion == 'entropy' else gini
    n = len(y)
    n_l, n_r = len(y_left), len(y_right)
    parent_impurity   = impurity_fn(y)
    weighted_children = (n_l / n) * impurity_fn(y_left) + (n_r / n) * impurity_fn(y_right)
    return parent_impurity - weighted_children


# Toy example: does splitting on x <= 2 help?
y = np.array([0, 0, 0, 1, 1, 1])
x = np.array([1, 2, 3, 4, 5, 6])

split_at = 2
y_l = y[x <= split_at]
y_r = y[x >  split_at]

ig = information_gain(y, y_l, y_r)
print(f'Split at x<=2: left={y_l}, right={y_r}')
print(f'Information Gain = {ig:.4f}')

# A perfect split should give IG = 0 (parent) - 0 (children) = parent entropy
ig_perfect = information_gain(y, np.array([0,0,0]), np.array([1,1,1]))
print(f'Perfect split IG = {ig_perfect:.4f}  (should equal parent entropy = {entropy(y):.4f})')

Split at x<=2: left=[0 0], right=[0 1 1 1]
Information Gain = 0.4591
Perfect split IG = 1.0000  (should equal parent entropy = 1.0000)


---
## Part 3 — Finding the Best Split

For each feature, we try every unique value as a potential threshold, compute IG, and keep the best.

**Key insight:** only *unique* values need to be checked as thresholds — or even just midpoints between consecutive sorted values.

In [5]:
def best_split(X, y, criterion='entropy'):
    """
    Search all features and thresholds for the split with the highest IG.
    
    Returns
    -------
    best_feature  : int   — column index of the best feature
    best_threshold: float — threshold value for the split
    best_ig       : float — information gain achieved
    """
    n_samples, n_features = X.shape
    best_ig        = -1
    best_feature   = None
    best_threshold = None

    for feature_idx in range(n_features):
        column     = X[:, feature_idx]
        thresholds = # candidate split points

        for threshold in thresholds:
            y_l   = # labels of points in left node
            y_r   = # labels of points in right node

            # Skip degenerate splits (all samples go one way)
            if len(y_l) == 0 or len(y_r) == 0:
                continue

            ig = information_gain(y, y_l, y_r, criterion)

            if ig > best_ig:
                best_ig        = ig
                best_feature   = feature_idx
                best_threshold = threshold

    return best_feature, best_threshold, best_ig


# Quick test on 2-feature toy data
X_toy = np.array([[2, 3],
                  [1, 4],
                  [5, 1],
                  [6, 2]])
y_toy = np.array([0, 0, 1, 1])

feat, thresh, ig = best_split(X_toy, y_toy)
print(f'Best split: feature {feat}, threshold {thresh}, IG = {ig:.4f}')
print('(A perfect split should have IG equal to the parent entropy)')

Best split: feature 0, threshold 2, IG = 1.0000
(A perfect split should have IG equal to the parent entropy)


### Exercise 2
Why do we skip a split when one side is empty? What would happen to the IG formula if we didn't?

*Write your answer here (double-click to edit):*

> ...

---
## Part 4 — The Tree Node

A decision tree is a **binary tree** where each internal node stores:
- which feature to split on
- the threshold value
- references to left and right children

Each leaf node stores the **majority class** of the training samples that reached it.

In [ ]:
class Node:
    """
    A single node in the decision tree.

    Internal node: feature, threshold, left, right are set.
    Leaf node:     value is set (predicted class).
    """
    def __init__(self,
                 feature=None, threshold=None,
                 left=None,    right=None,
                 value=None):
        self.feature   = feature     # feature index to split on
        self.threshold = threshold   # split threshold
        self.left      = left        # Node for x <= threshold
        self.right     = right       # Node for x >  threshold
        self.value     = value       # majority class (leaf only)

    def is_leaf(self):
        return self.value is not None


print('Node class defined.')

---
## Part 5 — Building the Tree

We build the tree **recursively**. At each call we either:
- **Stop** (base case) and create a leaf, OR
- **Split** and recurse on left and right subsets

### Stopping conditions (when to make a leaf):
1. All samples have the **same class** → entropy = 0, nothing to gain
2. We've hit the **maximum depth** limit
3. The node has **fewer samples than** `min_samples_split`
4. No split produces any information gain (IG = 0)

In [ ]:
class DecisionTreeClassifier:
    """
    A CART-style decision tree classifier built from scratch.

    Parameters
    ----------
    max_depth        : int   — maximum tree depth (None = unlimited)
    min_samples_split: int   — minimum samples required to attempt a split
    criterion        : str   — 'entropy' or 'gini'
    """

    def __init__(self, max_depth=None, min_samples_split=2, criterion='entropy'):
        self.max_depth         = max_depth
        self.min_samples_split = min_samples_split
        self.criterion         = criterion
        self.root              = None


    def fit(self, X, y):
        """Build the decision tree from training data X, y."""
        self.root = self._build(X, y, depth=0)
        return self

    def predict(self, X):
        """Predict class labels for each row in X."""
        return np.array([self._traverse(row, self.root) for row in X])

    def _majority_class(self, y):
        """Return the most common class in y."""
        counter = Counter(y)
        return counter.most_common(1)[0][0]

    def _build(self, X, y, depth):
        """
        Recursively build the tree.
        Returns a Node (either a leaf or an internal split node).
        """
        n_samples = len(y)
        n_classes  = len(np.unique(y))

        # ---- Stopping conditions ----
        if (n_classes == 1                               # pure node
                or #TODO    # too few samples
                or #TODO:
            return Node(value=self._majority_class(y))

        # ---- Find best split ----
        feat, thresh, ig = #TODO

        if feat is None or ig == 0:       # no useful split found
            return Node(value=self._majority_class(y))

        # ---- Partition and recurse ----
        mask   = X[:, feat] <= thresh
        left   = self._build(X[mask],  y[mask],  depth + 1)
        right  = self._build(X[~mask], y[~mask], depth + 1)

        return Node(feature=feat, threshold=thresh, left=left, right=right)

    def _traverse(self, x, node):
        """Walk down the tree for a single sample x."""
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse(x, node.left)
        else:
            return self._traverse(x, node.right)


print('DecisionTreeClassifier defined.')

---
## Part 6 — Test on a Toy Dataset

Before scaling up, let's verify the tree works on a dataset we can visualise completely.

In [ ]:
# XOR-like toy problem (not linearly separable)
X_toy = np.array([
    [0, 0], [0, 1], [1, 0], [1, 1],
    [0, 0], [0, 1], [1, 0], [1, 1],
])
y_toy = np.array([0, 1, 1, 0, 0, 1, 1, 0])

tree = DecisionTreeClassifier(max_depth=3)
tree.fit(X_toy, y_toy)

preds = tree.predict(X_toy)
print('Predictions:', preds)
print('Ground truth:', y_toy)
print('Training accuracy:', np.mean(preds == y_toy))

---
## Part 7 — Visualising the Decision Boundary

Let's generate a more interesting 2-D dataset and see what boundaries the tree draws.

In [ ]:
def make_circles_dataset(n=200, noise=0.1):
    """Two concentric circles — non-linear, good for visualising tree boundaries."""
    angles = np.linspace(0, 2 * np.pi, n // 2)
    inner  = np.column_stack([np.cos(angles) * 0.5, np.sin(angles) * 0.5])
    outer  = np.column_stack([np.cos(angles),        np.sin(angles)])
    X = np.vstack([inner, outer]) + np.random.randn(n, 2) * noise
    y = np.array([0] * (n // 2) + [1] * (n // 2))
    return X, y

X_c, y_c = make_circles_dataset(n=200, noise=0.1)

def plot_decision_boundary(model, X, y, title='Decision Boundary', ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    h = 0.02
    x_min, x_max = X[:, 0].min() - .3, X[:, 0].max() + .3
    y_min, y_max = X[:, 1].min() - .3, X[:, 1].max() + .3
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                          np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', edgecolors='k', s=20)
    ax.set_title(title)
    return ax

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, depth in zip(axes, [1, 3, 10]):
    t = DecisionTreeClassifier(max_depth=depth).fit(X_c, y_c)
    acc = np.mean(t.predict(X_c) == y_c)
    plot_decision_boundary(t, X_c, y_c,
                           title=f'max_depth={depth}  (train acc={acc:.2f})',
                           ax=ax)
plt.tight_layout()
plt.show()

**Observe:** As depth increases the tree can fit the training data more tightly — but it may start memorising noise (overfitting).

### Exercise 3
Re-run the cell above but use a train/test split. Plot *test* accuracy vs `max_depth` for depths 1–15. At what depth does the test accuracy peak?

In [ ]:
# Your code here
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.3, random_state=0)

depths       = range(1, 16)
train_accs   = []
test_accs    = []

for d in depths:
    t = DecisionTreeClassifier(max_depth=d).fit(X_tr, y_tr)
    train_accs.append(np.mean(t.predict(X_tr) == y_tr))
    test_accs.append( np.mean(t.predict(X_te) == y_te))

plt.figure(figsize=(7, 4))
plt.plot(depths, train_accs, label='Train accuracy', marker='o')
plt.plot(depths, test_accs,  label='Test accuracy',  marker='s')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Bias–Variance tradeoff in a Decision Tree')
plt.grid(True)
plt.show()

best_depth = depths[np.argmax(test_accs)]
print(f'Best test accuracy at depth {best_depth}: {max(test_accs):.3f}')

---
## Part 8 — Real Dataset: Iris

Now let's test our tree on a standard benchmark and compare with sklearn.

In [ ]:
from sklearn.tree import DecisionTreeClassifier as SklearnDT

iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris,
                                            test_size=0.2, random_state=42)

# Our tree
our_tree = DecisionTreeClassifier(max_depth=4, criterion='entropy').fit(X_tr, y_tr)
our_acc  = accuracy_score(y_te, our_tree.predict(X_te))

# Sklearn tree
sk_tree  = SklearnDT(max_depth=4, criterion='entropy').fit(X_tr, y_tr)
sk_acc   = accuracy_score(y_te, sk_tree.predict(X_te))

print(f'Our tree  test accuracy: {our_acc:.3f}')
print(f'Sklearn   test accuracy: {sk_acc:.3f}')

---
## Part 9 — Printing the Tree

A useful debugging tool: print the tree structure so you can trace how it reasons.

In [ ]:
def print_tree(node, feature_names=None, indent='', last=True):
    """Pretty-print the tree structure."""
    connector = '└── ' if last else '├── '
    if node.is_leaf():
        print(indent + connector + f'Leaf → class {node.value}')
    else:
        fname = (feature_names[node.feature]
                 if feature_names else f'X[{node.feature}]')
        print(indent + connector + f'{fname} <= {node.threshold:.3f}')
        child_indent = indent + ('    ' if last else '│   ')
        print_tree(node.left,  feature_names, child_indent, last=False)
        print_tree(node.right, feature_names, child_indent, last=True)


# Show the Iris tree (depth 3 for readability)
small_tree = DecisionTreeClassifier(max_depth=3, criterion='entropy').fit(X_tr, y_tr)
print('Decision Tree structure:')
print_tree(small_tree.root, feature_names=iris.feature_names)

---
## Part 10 — Feature Importance

A simple way to measure how useful each feature was: sum up the information gain at every split that used that feature, weighted by the number of samples.

$$\text{importance}(j) = \sum_{\text{nodes that split on } j} \frac{n_{\text{node}}}{n_{\text{total}}} \cdot \text{IG}$$

In [ ]:
def compute_feature_importances(tree, n_features, X_train, y_train):
    """
    Walk the tree and accumulate importance scores.
    
    NOTE: This is a simplified version — sklearn uses a more efficient
    approach that stores these values during training.
    """
    importances = np.zeros(n_features)
    n_total     = len(y_train)

    def _walk(node, mask):
        """Recurse; mask tracks which training samples reach this node."""
        if node.is_leaf():
            return
        y_node = y_train[mask]
        n_node = mask.sum()

        # Recompute IG for the split at this node
        col    = X_train[:, node.feature]
        left_mask  = mask & (col <= node.threshold)
        right_mask = mask & (col >  node.threshold)

        ig = information_gain(y_node,
                              y_train[left_mask],
                              y_train[right_mask],
                              tree.criterion)

        importances[node.feature] += (n_node / n_total) * ig
        _walk(node.left,  left_mask)
        _walk(node.right, right_mask)

    _walk(tree.root, np.ones(len(y_train), dtype=bool))

    # Normalise to sum to 1
    total = importances.sum()
    if total > 0:
        importances /= total
    return importances


our_tree4 = DecisionTreeClassifier(max_depth=4, criterion='entropy').fit(X_tr, y_tr)
importances = compute_feature_importances(our_tree4, X_tr.shape[1], X_tr, y_tr)

plt.figure(figsize=(7, 3))
plt.barh(iris.feature_names, importances, color='steelblue')
plt.xlabel('Normalised importance')
plt.title('Feature importances (our tree)')
plt.tight_layout()
plt.show()

print('\nFor reference — sklearn importances:')
for name, imp in zip(iris.feature_names, sk_tree.feature_importances_):
    print(f'  {name}: {imp:.3f}')

---
## Part 11 — Gini vs Entropy

Let's compare the two criteria side-by-side.

In [ ]:
# Compare Gini vs Entropy over a range of depths on Iris
depths = range(1, 15)
results = {}

for crit in ('entropy', 'gini'):
    accs = []
    for d in depths:
        t = DecisionTreeClassifier(max_depth=d, criterion=crit).fit(X_tr, y_tr)
        accs.append(accuracy_score(y_te, t.predict(X_te)))
    results[crit] = accs

plt.figure(figsize=(7, 4))
for crit, accs in results.items():
    plt.plot(depths, accs, marker='o', label=crit)
plt.xlabel('max_depth')
plt.ylabel('Test Accuracy (Iris)')
plt.title('Entropy vs Gini on Iris')
plt.legend()
plt.grid(True)
plt.show()

---
## Summary

Here's what we built, step by step:

| Step | What we implemented |
|------|----------------------|
| 1 | `entropy(y)` and `gini(y)` — impurity measures |
| 2 | `information_gain(y, y_l, y_r)` — quality of a split |
| 3 | `best_split(X, y)` — exhaustive search over features × thresholds |
| 4 | `Node` — the building block of the tree |
| 5 | `DecisionTreeClassifier._build()` — recursive tree construction |
| 6 | `DecisionTreeClassifier._traverse()` — prediction by following splits |
| 7 | `print_tree()` — human-readable tree display |
| 8 | `compute_feature_importances()` — which features did the tree rely on? |

### Key takeaways
- Trees split greedily — they don't backtrack to reconsider earlier choices.
- Deep trees overfit; use `max_depth` or `min_samples_split` to regularise.
- Gini and entropy produce very similar trees in practice.
- Feature importance tells you *which* features drove the decisions.

### What's next?
- **Pruning** (post-hoc depth reduction to improve generalisation)
- **Random Forests** (many trees on bootstrap samples — averages out variance)
- **Gradient Boosted Trees** (trees trained sequentially to fix residual errors)

---
## Challenge Exercises

1. **Regression trees.** Right now the leaf predicts the *majority class*. Modify the code so that for a continuous target $y$, the leaf stores the *mean* of $y$, and the split criterion minimises the weighted variance instead of entropy.

2. **Faster threshold search.** Instead of trying every unique value, try only midpoints between consecutive sorted values. Does this change any results? Is it faster?

3. **`min_samples_leaf`.** Add a new hyperparameter that requires each leaf to contain at least $k$ training samples. Where in `_build()` should this check go?

4. **Categorical features.** Our tree only handles numerical features with `<=` splits. How would you extend it to support categorical features (e.g. colour ∈ {red, green, blue})?

5. **Visualise the tree with matplotlib.** Write a function that draws the tree as a diagram (boxes and arrows) rather than text.